In [25]:
"""
Setup for propagating through turbulent atmosphere. This consists of defining the propagation geometry and turbulent properties.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, Bounds

#### Determine Geometry ####
D2 = 0.5            # Diameter of the observation plane [m]
wvl = 1e-6          # Wavelength [m]
k = 2*np.pi/wvl     # Wavenumber [rad/m]
Dz = 50e3           # Propagation distance [m]

#### Point Source Properties ####
DROI = 4*D2         # Diameter of region of interest [m]
D1 = wvl*Dz/DROI    # Diameter of central lobe [m]
R = Dz              # Wavefront radius of curvature [m]

#### Atmospheric Properties ####
Cn2 = 1e-16         # Refractive index structure parameter [m^-2/3]
r0sw = (0.423*k**2*Cn2*3/8*Dz)**(-3/5)  # Spherical wave coherence diameter [m]
r0pw = (0.423*k**2*Cn2*Dz)**(-3/5)      # Plane wave coherence diameter [m]
p = np.linspace(0,Dz,int(1e3))

#### Log-Amplitude Variance ####
rytov = 0.563*k**(7/6)*np.sum(Cn2*(1-p/Dz)**(5/6)\
                              *p**(5/6)*(p[1]-p[0]))    # Equation 9.69 (J.Schmidt)

#### Screen Properties ####
nscr = 11
A = np.zeros((2,nscr))
alpha = np.arange(0,nscr)/(nscr-1)
A[0,:] = alpha**(5/3)
A[1,:] = (1-alpha)**(5/6) * alpha**(5/6)
b = np.array([[r0sw**(-5/3)],[rytov/1.33*(k/Dz)**(5/6)]])

#### Initial Guess ####
x0 = (nscr/3 * r0sw * np.ones((nscr,1)))**(-5/3)
fun = lambda X: np.sum((A*X.flatten(order='F')-b)**2)
x1 = np.zeros((nscr,1))
rmax = 0.1
x2 = rmax/1.33*(k/Dz)**(5/6)/A[1,:]
x2[A[1,:] == 0] = 50**(-5/3)
print(f"x1 = {x1}")
print(f"x2 = {x2}")
bounds = Bounds(x1.flatten(),x2)
min_results = minimize(fun,x0.flatten(),method='trust-constr',bounds=bounds)
r0scrn = min_results.x**(-3/5)
r0scrn[np.isinf(r0scrn)] = 1e6
print(f"r0scrn = {r0scrn}")


x1 = [[0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]
x2 = [1.47361260e-03 3.14017960e+01 1.94412086e+01 1.54991245e+01
 1.38669371e+01 1.34031406e+01 1.38669371e+01 1.54991245e+01
 1.94412086e+01 3.14017960e+01 1.47361260e-03]
r0scrn = [75.78375152  0.12642343  0.16856454  0.19311493  0.20644853  0.21070559
  0.20644852  0.19311489  0.1685645   0.12642339 50.04844604]


/tmp/ipykernel_7424/61693146.py:43: RuntimeWarning: divide by zero encountered in divide
  x2 = rmax/1.33*(k/Dz)**(5/6)/A[1,:]
